# Preprocesado de datos y división del dataset

En este notebook vamos a preparar los datos para el entrenamiento del modelo clasificador de riesgo IA (AI Act).

Los pasos que seguiremos son:
1. Carga del dataset limpio
2. Aplicar NER (Named Entity Recognition) a la columna `descripcion`
3. Explorar las entidades extraídas por tipo y clase de riesgo
4. Preprocesar el texto con lematización
5. Dividir en train / validation / test (70% / 15% / 15%) con estratificación
6. Guardar los conjuntos en `data/processed/`

In [1]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_dataset_artificial"))
        break

import functions  # noqa: E402
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_ia_artificial"
functions._DATASET_TAGS = {"dataset_type": "artificial", "dataset_source": "eu_ai_act_flagged"}

In [2]:
#Al principio de cada notebook, añadimos estas líneas para que se recarguen automáticamente las funciones que hayamos editado en el módulo functions.py sin tener que reiniciar el kernel cada vez.
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd

path = "datasets/dataset_riesgo_limpio.csv"
df = pd.read_csv(path)

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")
print("\nDistribución de clases:")
print(df["etiqueta"].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/dataset_riesgo_limpio.csv'

## 1. Aplicar NER al dataset

In [ ]:
from functions import extraer_entidades, resumen_entidades

# Aplicamos NER a la columna de descripción original (sin limpiar)
df = extraer_entidades(df, "descripcion")

# Mostramos las entidades de las primeras 3 filas como ejemplo
for i in range(3):
    print(f"Texto: {df['descripcion'].iloc[i][:100]}...")
    print(f"Entidades: {df['entidades'].iloc[i]}")
    print()

Texto: Sistema de puntuación social que clasifica a ciudadanos según su historial de pagos, multas y compor...
Entidades: []

Texto: Sistema de identificación biométrica facial en tiempo real en cámaras de metro y autobús urbano para...
Entidades: []

Texto: Plataforma que usa técnicas subliminales en anuncios online para manipular las decisiones de compra ...
Entidades: []



In [ ]:
# Resumen de entidades por tipo y por clase de riesgo
freq_por_tipo, freq_por_tipo_clase = resumen_entidades(df)

Frecuencia global de entidades por tipo:
tipo_entidad
MISC    50
LOC     30
PER     22
ORG      6
Name: count, dtype: int64

Frecuencia de entidades por tipo y clase de riesgo:
tipo_entidad     LOC  MISC  ORG  PER
clase                               
alto_riesgo        5    21    4    8
inaceptable        7     8    0    1
riesgo_limitado   17     4    1   12
riesgo_minimo      1    17    1    1


### Conclusión del NER

Las entidades nombradas (NER) en este dataset regulatorio no aportan señal discriminativa significativa para la clasificación de riesgo. Los textos describen sistemas genéricos de IA sin mencionar nombres propios, organizaciones o ubicaciones específicas que diferencien las clases.

Por tanto, **continuamos el entrenamiento utilizando solo el texto limpio y lematizado** como feature principal.

## 2. Preprocesado del texto con lematización

In [ ]:
from functions import preparar_dataset

# preparar_dataset ahora usa limpiar_texto_preprocess (con lematización)
df_processed = preparar_dataset(df, "descripcion", "etiqueta")

print(f"Dataset procesado: {df_processed.shape}")
print("\nEjemplo de texto lematizado:")
print(f"  Original:   {df['descripcion'].iloc[0][:100]}...")
print(f"  Lematizado: {df_processed['text_final'].iloc[0][:100]}...")

Dataset procesado: (300, 2)

Ejemplo de texto lematizado:
  Original:   Sistema de puntuación social que clasifica a ciudadanos según su historial de pagos, multas y compor...
  Lematizado: sistema puntuación social clasificar ciudadano historial pago multa comportamiento red restringir ac...


## 3. División en train / validation / test

In [ ]:
from functions import split_dataset

# Dividimos en train (70%), validation (15%) y test (15%) con estratificación
train_df, val_df, test_df = split_dataset(df_processed, "etiqueta")

print("\nDistribución por clase en cada conjunto:")
print("\nTrain:")
print(train_df["etiqueta"].value_counts())
print("\nValidation:")
print(val_df["etiqueta"].value_counts())
print("\nTest:")
print(test_df["etiqueta"].value_counts())

Train: 210 muestras
Validation: 45 muestras
Test: 45 muestras

Distribución por clase en cada conjunto:

Train:
etiqueta
alto_riesgo        63
inaceptable        54
riesgo_limitado    47
riesgo_minimo      46
Name: count, dtype: int64

Validation:
etiqueta
alto_riesgo        14
inaceptable        11
riesgo_limitado    10
riesgo_minimo      10
Name: count, dtype: int64

Test:
etiqueta
alto_riesgo        13
inaceptable        12
riesgo_limitado    10
riesgo_minimo      10
Name: count, dtype: int64


## 4. Guardar los conjuntos procesados

In [ ]:
import os

output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(os.path.join(output_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(output_dir, "validation.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "test.csv"), index=False)

print(f"Archivos guardados en {output_dir}/:")
for f in os.listdir(output_dir):
    print(f"  {f}")

Archivos guardados en data/processed/:
  test.csv
  train.csv
  validation.csv


In [ ]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
from functions import log_mlflow_safe

try:
    log_mlflow_safe(
        run_name="preprocesado",
        params={
            "lemmatize":    True,
            "ner":          True,
            "test_size":    0.15,
            "val_size":     0.15,
            "random_state": 42,
        },
        metrics={
            "n_train": len(train_df),
            "n_val":   len(val_df),
            "n_test":  len(test_df),
        },
    )
    print("✓ Preprocesado registrado en MLflow")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://18.201.64.41/


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\r

✓ Run 'preprocesado' registrado en MLflow (https://18.201.64.41/)
✓ Preprocesado registrado en MLflow


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
